In [ ]:
import pandas as pd
import time
import hopsworks
import os
from datetime import date
from dotenv import load_dotenv
import re
from features_utils import clean_and_format_columns, add_rolling_features
import utils
from weekly_feature_utils import *

/home/hugo/projects/player_stats_predictor/footenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded teams from JSON: [{'code': '18bb7c10', 'name': 'Arsenal'}, {'code': 'b8fd03ef', 'name': 'Manchester-City'}, {'code': '8602292d', 'name': 'Aston-Villa'}, {'code': 'cff3d9bb', 'name': 'Chelsea'}, {'code': '47c64c55', 'name': 'Crystal-Palace'}, {'code': '822bd0ba', 'name': 'Liverpool'}, {'code': '8ef52968', 'name': 'Sunderland'}, {'code': '19538871', 'name': 'Manchester-United'}, {'code': 'd3fd31cc', 'name': 'Everton'}, {'code': 'd07537b9', 'name': 'Brighton-and-Hove-Albion'}, {'code': '361ca564', 'name': 'Tottenham-Hotspur'}, {'code': 'b2b47a98', 'name': 'Newcastle-United'}, {'code': 'fd962109', 'name': 'Fulham'}, {'code': 'cd051869', 'name': 'Brentford'}, {'code': '4ba7cbea', 'name': 'Bournemouth'}, {'code': 'e4a775cb', 'name': 'Nottingham-Forest'}, {'code': '5bfb9659', 'name': 'Leeds-United'}, {'code': '7c21e445', 'name': 'West-Ham-United'}, {'code': '943e8050', 'name': 'Burnley'}, {'code': '8cec06e1', 'name': 'Wolverhampton-Wanderers'}]


In [2]:
season = get_current_season()
project_name = "player_stat_prediction"
fg_name = "player_stats_rolling"

# 2. Get Historical Data
df_history, fg = get_hopsworks_data(project_name, fg_name)
existing_match_ids = set(df_history['match_id'].unique()) if not df_history.empty else set()

# 3. Filter New URLs
all_match_urls = utils.get_all_season_match(season)

new_matches_to_scrape = []
for url in all_match_urls:
    target_id = format_url_to_hopsworks_id(url)
    if target_id not in existing_match_ids:
        new_matches_to_scrape.append((url, target_id))

if not new_matches_to_scrape:
    print("No new matches to process.")

print("Number of new matches to process:", len(new_matches_to_scrape))
# 4. Scrape and Assign Formatted ID
new_data_list = []
for url, target_id in new_matches_to_scrape:
    print(f"Processing: {target_id}")
    
    try:
        # Parse Team Names from the target_id
        id_parts = target_id.split('_')
        home_team_name = id_parts[0]
        away_team_name = id_parts[1]

        # Use your utils function
        df_home, df_away = utils.stat_player_match(url)
        
        if df_home is not None and df_away is not None:
            # Add metadata
            df_home['team'] = home_team_name
            df_away['team'] = away_team_name
            
            # Combine match data
            combined_match_df = pd.concat([df_home, df_away], ignore_index=True)
            combined_match_df['match_id'] = target_id
            
            # Clean columns (standardize names for Hopsworks)
            clean_df = clean_and_format_columns(combined_match_df)
            new_data_list.append(clean_df)
            # Respectful delay between requests
            time.sleep(6) 
        else:
            print(f"Skipping {target_id}: Missing one or both team tables.")

    except Exception as e:
        print(f"Failed to process {target_id}: {e}")


2026-01-11 17:45:49,902 INFO: Initializing external client
2026-01-11 17:45:49,902 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-01-11 17:45:50,849 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/3195
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (5.55s) 
Loaded teams from JSON: [{'code': '18bb7c10', 'name': 'Arsenal'}, {'code': 'b8fd03ef', 'name': 'Manchester-City'}, {'code': '8602292d', 'name': 'Aston-Villa'}, {'code': 'cff3d9bb', 'name': 'Chelsea'}, {'code': '47c64c55', 'name': 'Crystal-Palace'}, {'code': '822bd0ba', 'name': 'Liverpool'}, {'code': '8ef52968', 'name': 'Sunderland'}, {'code': '19538871', 'name': 'Manchester-United'}, {'code': 'd3fd31cc', 'name': 'Everton'}, {'code': 'd07537b9', 'name': 'Brighton-and-Hove-Albion'}, {'code': '361ca564', 'name': 'Tottenham-Hotspur'}, {'code': 'b2b47a98', 'name': 'Newcastle-United'}, {'code': 'fd962109', 'name': 'Fulham'}, {'code'

✅ DataFrame pour team number 1 extrait et nettoyé.
---
✅ DataFrame pour team number 2 extrait et nettoyé.
---
Processing: Newcastle United_Crystal Palace_January 4, 2026
Request successful, parsing HTML...


✅ DataFrame pour team number 1 extrait et nettoyé.
---
✅ DataFrame pour team number 2 extrait et nettoyé.
---
Processing: Chelsea_Arsenal_November 30, 2025
Request successful, parsing HTML...
✅ DataFrame pour team number 1 extrait et nettoyé.
---
✅ DataFrame pour team number 2 extrait et nettoyé.
---


Processing: Everton_Brentford_January 4, 2026
Request successful, parsing HTML...


✅ DataFrame pour team number 1 extrait et nettoyé.
---
✅ DataFrame pour team number 2 extrait et nettoyé.
---
Processing: Brighton and Hove Albion_Fulham_August 16, 2025
Request successful, parsing HTML...


...

✅ DataFrame pour team number 1 extrait et nettoyé.
---
✅ DataFrame pour team number 2 extrait et nettoyé.
---
Processing: Arsenal_Tottenham Hotspur_November 23, 2025
Request successful, parsing HTML...


✅ DataFrame pour team number 1 extrait et nettoyé.
---
✅ DataFrame pour team number 2 extrait et nettoyé.
---
Processing: Brighton and Hove Albion_Burnley_January 3, 2026
Request successful, parsing HTML...


✅ DataFrame pour team number 1 extrait et nettoyé.
---
✅ DataFrame pour team number 2 extrait et nettoyé.
---
Processing: Fulham_Liverpool_January 4, 2026
Request successful, parsing HTML...


✅ DataFrame pour team number 1 extrait et nettoyé.
---
✅ DataFrame pour team number 2 extrait et nettoyé.
---
Processing: Brighton and Hove Albion_Tottenham Hotspur_September 20, 2025
Request successful, parsing HTML...


✅ DataFrame pour team number 1 extrait et nettoyé.
---
✅ DataFrame pour team number 2 extrait et nettoyé.
---


In [34]:
def rename_duplicates(df):
    cols = pd.Series(df.columns)
    for dup in cols[cols.duplicated()].unique(): 
        cols[cols == dup] = [f"{dup}_{i}" if i != 0 else dup for i in range(sum(cols == dup))]
    df.columns = cols
    return df

if new_data_list:
    for i, df in enumerate(new_data_list):
        df = rename_duplicates(df)
        df = df[~df['player'].str.contains("Players", na=False)]
        
        # IMPORTANT: Update the list with the cleaned dataframe
        new_data_list[i] = df
    # 1. Combine the dataframes
    df_new_batch = pd.concat(new_data_list, ignore_index=True)
    # df_new_batch = rename_duplicates(df_new_batch)
    base_cols = df_new_batch.columns.tolist()
    combined_df = pd.concat([df_history[base_cols], df_new_batch], axis=0)

    print("test", combined_df.columns)
    # 2. Run your function on the combined dataset
    # This ensures the window 'sees' the previous matches in df_history
    combined_df_with_rolling = add_rolling_features(combined_df, window_size=6)

    # 3. Filter back to just the new data
    # We use the match_ids from your new list to isolate the fresh rows
    new_match_ids = new_data_list[0]['match_id'].unique()
    final_new_data = combined_df_with_rolling[combined_df_with_rolling['match_id'].isin(df_new_batch['match_id'].unique())]

    print(final_new_data.head())

test Index(['player', 'player_number', 'nation', 'pos', 'age', 'min', 'gls', 'ast',
       'pk', 'pkatt', 'sh', 'sot', 'crdy', 'crdr', 'touches', 'tkl', 'int',
       'blocks', 'xg', 'npxg', 'xag', 'sca', 'gca', 'cmp', 'att', 'cmp_perc',
       'prgp', 'carries', 'prgc', 'att_1', 'succ', 'team', 'match_id'],
      dtype='object')
                 player  player_number   nation    pos     age  min  gls  ast  \
787        Aaron Hickey            2.0  sct SCO     LB  23-165   11    0    0   
114        Aaron Hickey            2.0  sct SCO     LB  23-208   75    0    0   
805        Aaron Hickey            2.0  sct SCO     RB  23-211    9    0    0   
995   Aaron Wan-Bissaka           29.0   cd COD  WB,RB  28-011   90    0    0   
297  Abdukodir Khusanov           45.0   uz UZB     CB  21-183   84    0    0   

     pk  pkatt  ...  rolling_avg_sca  rolling_avg_gca  rolling_avg_cmp  \
787   0      0  ...         0.500000              0.0        16.500000   
114   0      0  ...         0.500

In [ ]:
# try to push to hopsworks
if final_new_data:
    fg.insert(final_new_data, write_options={"wait_for_job": True})

Uploading Dataframe: 100.00% |██████████| Rows 1332/1332 | Elapsed Time: 00:00 | Remaining Time: 00:00


Launching job: player_stats_rolling_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://eu-west.cloud.hopsworks.ai:443/p/3195/jobs/named/player_stats_rolling_1_offline_fg_materialization/executions
2026-01-11 18:22:20,983 INFO: Waiting for execution to finish. Current state: SUBMITTED. Final status: UNDEFINED
2026-01-11 18:22:24,153 INFO: Waiting for execution to finish. Current state: RUNNING. Final status: UNDEFINED
2026-01-11 18:24:40,651 INFO: Waiting for execution to finish. Current state: SUCCEEDING. Final status: UNDEFINED
2026-01-11 18:24:43,791 INFO: Waiting for execution to finish. Current state: AGGREGATING_LOGS. Final status: SUCCEEDED
2026-01-11 18:24:43,903 INFO: Waiting for log aggregation to finish.
2026-01-11 18:24:52,451 INFO: Execution finished successfully.


(Job('player_stats_rolling_1_offline_fg_materialization', 'PYSPARK'), None)